In [5]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# 1. Fetch AI4I 2020 dataset from UCI (Dataset ID is 601)
ai4i = fetch_ucirepo(id=601)

# 2. Combine sensor columns (features) and failure labels (targets) into one table
df = pd.concat([ai4i.data.features, ai4i.data.targets], axis=1)

# 3. Save a copy as a CSV file in your project folder
df.to_csv("ai4i2020.csv", index=False)

# 4. Show the first 5 rows in the notebook
df.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
# Define the 5 live telemetry sensors we care about
sensor_cols = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear"
]

# Split the dataset into healthy rows (0) and broken rows (1)
normal_data = df[df["Machine failure"] == 0][sensor_cols]
faulty_data = df[df["Machine failure"] == 1][sensor_cols]

# Also grab a specific failure type: Heat Dissipation Failure (HDF)
hdf_faults = df[df["HDF"] == 1][sensor_cols]

print(f"Healthy rows (Normal): {len(normal_data)}")
print(f"Total Broken rows (All Faults): {len(faulty_data)}")
print(f"Heat Dissipation Fault rows (HDF): {len(hdf_faults)}")

Healthy rows (Normal): 9661
Total Broken rows (All Faults): 339
Heat Dissipation Fault rows (HDF): 115


In [9]:
normal_data.head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,298.1,308.6,1551,42.8,0
1,298.2,308.7,1408,46.3,3
2,298.1,308.5,1498,49.4,5
3,298.2,308.6,1433,39.5,7
4,298.2,308.7,1408,40.0,9


In [10]:
faulty_data.head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
50,298.9,309.1,2861,4.6,143
69,298.9,309.0,1410,65.7,191
77,298.8,308.9,1455,41.3,208
160,298.4,308.2,1282,60.7,216
161,298.3,308.1,1412,52.3,218


In [11]:
# Show 5 rows where a Heat Dissipation Failure (HDF) actually happened
df[df["HDF"] == 1][["Air temperature", "Process temperature", "Rotational speed", "Torque", "Tool wear", "Machine failure", "HDF", "PWF", "OSF", "TWF"]].head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,HDF,PWF,OSF,TWF
3236,300.8,309.4,1342,62.4,113,1,1,0,0,0
3760,302.3,310.9,1377,46.8,166,1,1,0,0,0
3787,302.3,310.8,1377,47.3,22,1,1,0,0,0
3793,302.3,310.9,1379,49.3,34,1,1,0,0,0
3806,302.3,310.9,1360,44.0,67,1,1,0,0,0


In [14]:
from pyod.models.ecod import ECOD
import numpy as np
import json

# 1. Fit the Anomaly Model ONLY on healthy normal data
clf = ECOD(contamination=0.03)
clf.fit(normal_data)

# 2. Calculate baseline averages for our explanation logic
baseline_mean = normal_data.mean().values
baseline_std = normal_data.std().values

# 3. Simulate receiving 1 faulty row from your HDF test set
sample_row = hdf_faults.iloc[0].values

# 4. Get the model's prediction (0 = normal, 1 = anomaly) and score (0.0 to 1.0)
sample_input = np.asarray(sample_row).reshape(1, -1)

is_anomaly = int(clf.predict(sample_input)[0])
anomaly_score = float(clf.predict_proba(sample_input)[0][1])


# 5. Calculate Z-scores to explain WHICH sensors caused the anomaly
z_scores = np.abs((sample_row - baseline_mean) / baseline_std)
ranked_indices = np.argsort(z_scores)[::-1]

primary_sensor = sensor_cols[ranked_indices[0]]
secondary_sensor = sensor_cols[ranked_indices[1]]

# 6. Build the final Alert JSON payload for the Agent service
alert_payload = {
    "machine_id": "CNC_MILL_01",
    "is_anomaly": bool(is_anomaly),
    "anomaly_score": round(anomaly_score, 3),
    "explanation": {
        "primary_trigger": primary_sensor,
        "primary_z_score": round(float(z_scores[ranked_indices[0]]), 2),
        "secondary_trigger": secondary_sensor,
        "secondary_z_score": round(float(z_scores[ranked_indices[1]]), 2),
        "summary": f"Alert: {primary_sensor} and {secondary_sensor} deviated abnormally from normal baseline."
    }
}

# Print the clean JSON output
print(json.dumps(alert_payload, indent=2))

{
  "machine_id": "CNC_MILL_01",
  "is_anomaly": false,
  "anomaly_score": 0.269,
  "explanation": {
    "primary_trigger": "Torque",
    "primary_z_score": 2.4,
    "secondary_trigger": "Rotational speed",
    "secondary_z_score": 1.18,
    "summary": "Alert: Torque and Rotational speed deviated abnormally from normal baseline."
  }
}
